# Deepfake Classifier — Fine-Tuning
This notebook fine-tunes the `facebook/dinov2-base` model on the
`zguo0525/google-landmarks-v2-mini` dataset and saves the best checkpoint
to `./fine_tuned_model`.

It then saves a compact **weight delta** to
`./fine_tuned_model_delta/weight_delta.pt` (~4–15 MB) instead of committing
the full model weights to Git.

The delta stores only the parameter *differences* from the base model. At
inference time, `DeepfakeClassifier(landmark_delta_path=...)` re-downloads
the base model from HuggingFace (cached locally after the first run) and
applies the delta on top.

Run **`deepfake_classifier_eval.ipynb`** afterwards to compare the original
vs. fine-tuned model and evaluate the full integrated `DeepfakeClassifier`.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
from deepfake_classifier import fine_tune_model, save_weight_delta

## Fine-Tune the DINOv2 Landmark Model
This will:
1. Download `zguo0525/google-landmarks-v2-mini` (~2.4 GB)
2. Load `facebook/dinov2-base` as the backbone (classifier head re-initialised)
3. Train for 3 epochs (MPS-accelerated on Apple Silicon)
4. Save the best checkpoint to `./fine_tuned_model`

> ⚠️ **Note:** Training ~20 K images × 3 epochs takes roughly 60–90 minutes on
> an M-series MacBook. Make sure you have at least 16 GB of unified memory available.

In [3]:
model, processor = fine_tune_model(
    base_model_name="facebook/dinov2-base",
    dataset_name="zguo0525/google-landmarks-v2-mini",
    output_dir="./fine_tuned_model",
    epochs=3,
    batch_size=8,
    learning_rate=5e-5
)

Loading dataset: zguo0525/google-landmarks-v2-mini
Number of landmark classes: 3103
Applying transformations...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[transformers] Dinov2ForImageClassification LOAD REPORT from: facebook/dinov2-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,8.083845,8.053489,0.000967,0.000006,0.000003,0.000967
2,7.967538,7.936789,0.001611,0.000055,0.000028,0.001611
3,7.821877,7.838536,0.002900,0.000089,0.000046,0.002900


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =          3.0
  total_flos               = 4654310078GF
  train_loss               =       8.0006
  train_runtime            =   2:47:45.92
  train_samples_per_second =        6.018
  train_steps_per_second   =        0.188


## Save Weight Delta
Instead of committing the full model weights to Git, we save only the
*difference* between the fine-tuned weights and the base model weights.
The resulting file is typically **4–15 MB** and is safe to push to GitHub.

> The full `./fine_tuned_model/` directory is listed in `.gitignore`;
> only `./fine_tuned_model_delta/weight_delta.pt` is tracked by Git.

In [8]:
delta_path, size_mb = save_weight_delta(
    fine_tuned_model=model,
    base_model_name="facebook/dinov2-base",
    output_path="./fine_tuned_model_delta/weight_delta.pt",
)

print(f"\n✅ Delta saved to '{delta_path}' ({size_mb:.2f} MB) — safe to commit to Git.")

Loading base model 'facebook/dinov2-base' to compute delta...


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[transformers] Dinov2ForImageClassification LOAD REPORT from: facebook/dinov2-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, mps:0 and cpu!

## Training Complete
- **Full model** saved locally to `./fine_tuned_model/` (gitignored — do **not** commit).
- **Weight delta** saved to `./fine_tuned_model_delta/weight_delta.pt` (tracked by Git).
- Run `deepfake_classifier_eval.ipynb` to evaluate and compare both models.